# DBFS Demo Notebook

This notebook walks through the DBFS exercise step by step.

## Objectives
- Create a folder
- Create and list a file
- Read the CSV into Spark
- Rename the folder
- Update the file
- Clean up the folder

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

student_name = 'demo_student'
base_path = f'dbfs:/tmp/{student_name}/'
file_path = base_path + 'sample.csv'
print(base_path)
dbutils.fs.mkdirs(base_path)

## Step 1: Create a CSV file

In [ ]:
csv_text = "id,name,amount\n1,Alice,100\n2,Bob,200\n3,Charlie,300\n"
dbutils.fs.put(file_path, csv_text, overwrite=True)
print('Created file:', file_path)
print(dbutils.fs.ls(base_path))

## Step 2: Read the file into a Spark DataFrame

In [ ]:
df = spark.read.option('header', 'true').csv(file_path)
display(df)
df.createOrReplaceTempView('demo_table')
spark.sql("SELECT * FROM demo_table WHERE amount > 150").show()

## Step 3: Rename the folder

In [ ]:
new_path = f'dbfs:/tmp/{student_name}_renamed/'
dbutils.fs.mv(base_path, new_path)
print('Moved folder to:', new_path)
print(dbutils.fs.ls('dbfs:/tmp'))

## Step 4: Append a row and re-read the file

In [ ]:
updated_path = new_path + 'sample.csv'
dbutils.fs.put(updated_path, '\n4,Dan,250', overwrite=False)
df2 = spark.read.option('header', 'true').csv(updated_path)
display(df2)
df2.createOrReplaceTempView('demo_table_updated')
spark.sql("SELECT * FROM demo_table_updated").show()

## Step 5: Clean up

In [ ]:
dbutils.fs.rm(updated_path, recurse=False)
dbutils.fs.rm(new_path, recurse=True)
print('Cleanup complete.')